# Prompt Evaluation Dataset Generation with Gemini

This notebook demonstrates how to use Gemini to generate an evaluation dataset for various coding and configuration tasks.

In [1]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import display, Markdown

# 1. Load API Key from .env
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

# 2. Initialize the Client
client = genai.Client(api_key=api_key)
model_id = "gemini-2.5-flash"

In [2]:
# Helper functions for Gemini
def chat(prompt, system_instruction=None, temperature=1.0):
    """
    Simple wrapper for generating content with Gemini.
    """
    config = types.GenerateContentConfig(
        temperature=temperature,
        system_instruction=system_instruction
    )
    
    response = client.models.generate_content(
        model=model_id,
        contents=prompt,
        config=config
    )
    return response.text

In [3]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing a task that requires Python, JSON, or a Regex to complete.

Example output structure:
[
    {
        "task": "Description of task",
        "type": "python|json|regex"
    },
    ...
]

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks related to AWS services (S3, Lambda, EC2, etc.).
* Please generate 5 unique tasks.
"""
    
    print("Generating dataset...")
    response_text = chat(
        prompt=prompt, 
        system_instruction="You are a helpful assistant that outputs ONLY valid raw JSON.",
        temperature=0.7
    )
    
    # Clean response in case model includes markdown formatting
    clean_json = response_text.replace("```json", "").replace("```", "").strip()
    
    try:
        dataset = json.loads(clean_json)
        
        # Ensure data directory exists
        os.makedirs("../data", exist_ok=True)
        
        file_path = "../data/eval_dataset.json"
        with open(file_path, "w") as f:
            json.dump(dataset, f, indent=4)
            
        print(f"Dataset successfully saved to {file_path}")
        return dataset
    except Exception as e:
        print(f"Error parsing or saving JSON: {e}")
        print("Raw Response:", response_text)
        return None

In [4]:
# Execute the generation
dataset = generate_dataset()
if dataset:
    display(dataset)

Generating dataset...
Dataset successfully saved to ../data/eval_dataset.json


[{'task': "Write a Python Lambda handler function that stops an EC2 instance specified by an instance ID passed in the event payload. The instance ID should be extracted from the 'instance_id' key in the event.",
  'type': 'python'},
 {'task': "Generate a CloudFormation JSON resource definition for an S3 bucket named 'my-application-logs' that has public access blocked for all settings (BlockPublicAcls, IgnorePublicAcls, BlockPublicPolicy, RestrictPublicBuckets).",
  'type': 'json'},
 {'task': "Provide a regular expression to extract the AWS Account ID (a 12-digit number) from a string like 'arn:aws:iam::123456789012:user/example-user' or 'Account 123456789012 accessed S3 bucket'.",
  'type': 'regex'},
 {'task': 'Create a Python function using boto3 that takes an S3 bucket name and a file key as input, and uploads a local file at a given path to that S3 location.',
  'type': 'python'},
 {'task': "Generate an IAM policy document (JSON) that grants read-only access (s3:GetObject, s3:List